# Inspect Gold Star Schema

Notebook para validar a Gold dimensional da V2.

Por padrao, tenta abrir primeiro a amostra `dev`. Se ela nao existir, tenta abrir a Gold oficial.

In [56]:
from v2.config.paths import DELTA_ROOT, star_schema_gold_dir

official_root = star_schema_gold_dir(2025)
dev_root = DELTA_ROOT / "dev" / "gold" / "star_schema" / "2025_01"

root = dev_root if (dev_root / "fact_trips" / "_delta_log").exists() else official_root
table_names = ["dim_data", "dim_clima", "dim_localizacao", "fact_trips"]

print(f"Official root: {official_root}")
print(f"Dev root     : {dev_root}")
print(f"Using root   : {root}")

missing_tables = [
    table_name
    for table_name in table_names
    if not (root / table_name / "_delta_log").exists()
]

if missing_tables:
    raise FileNotFoundError(
        f"Tabelas Gold nao encontradas em {root}: {missing_tables}"
    )

Official root: /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/gold/star_schema/2025
Dev root     : /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/dev/gold/star_schema/2025_01
Using root   : /home/delldev/projetos/nyc-taxi-lakehouse/v2/data/delta/dev/gold/star_schema/2025_01


In [57]:
from v2.config.spark import create_spark

spark = create_spark("NotebookInspectGoldStarSchema")

In [58]:
dfs = {
    table_name: spark.read.format("delta").load(str(root / table_name))
    for table_name in table_names
}

dim_data = dfs["dim_data"]
dim_clima = dfs["dim_clima"]
dim_localizacao = dfs["dim_localizacao"]
fact_trips = dfs["fact_trips"]

In [59]:
for table_name, df in dfs.items():
    print(f"\n{table_name}")
    df.printSchema()


dim_data
root
 |-- data_id: integer (nullable = true)
 |-- data: date (nullable = true)
 |-- ano: integer (nullable = true)
 |-- mes: integer (nullable = true)
 |-- dia_mes: integer (nullable = true)
 |-- dia_semana_num: integer (nullable = true)
 |-- dia_semana_nome: string (nullable = true)
 |-- fim_de_semana: boolean (nullable = true)


dim_clima
root
 |-- clima_id: integer (nullable = true)
 |-- data: date (nullable = true)
 |-- fonte_clima: string (nullable = true)
 |-- escopo_clima: string (nullable = true)
 |-- qtd_estacoes: long (nullable = true)
 |-- qtd_estacoes_completas: long (nullable = true)
 |-- cobertura_estacoes_pct: double (nullable = true)
 |-- qtd_estacoes_com_precipitacao: long (nullable = true)
 |-- qtd_estacoes_com_temperatura: long (nullable = true)
 |-- precipitacao_media_mm: double (nullable = true)
 |-- precipitacao_max_mm: double (nullable = true)
 |-- temp_max_media_c: double (nullable = true)
 |-- temp_min_media_c: double (nullable = true)
 |-- temp_media

In [60]:
summary = spark.createDataFrame(
    [(table_name, df.count()) for table_name, df in dfs.items()],
    ["tabela", "linhas"],
)

summary.show(truncate=False)

+---------------+------+
|tabela         |linhas|
+---------------+------+
|dim_data       |365   |
|dim_clima      |365   |
|dim_localizacao|265   |
|fact_trips     |97065 |
+---------------+------+



## Cobertura de datas

In [61]:
from pyspark.sql import functions as F

dim_data.select(
    F.count("*").alias("linhas"),
    F.expr("count(distinct data)").alias("dias_distintos"),
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max"),
).show(truncate=False)

dim_clima.select(
    F.count("*").alias("linhas"),
    F.expr("count(distinct data)").alias("dias_distintos"),
    F.min("data").alias("data_min"),
    F.max("data").alias("data_max"),
    F.sum(F.col("registro_clima_incompleto").cast("int")).alias(
        "dias_clima_incompleto"
    ),
).show(truncate=False)

+------+--------------+----------+----------+
|linhas|dias_distintos|data_min  |data_max  |
+------+--------------+----------+----------+
|365   |365           |2025-01-01|2025-12-31|
+------+--------------+----------+----------+

+------+--------------+----------+----------+---------------------+
|linhas|dias_distintos|data_min  |data_max  |dias_clima_incompleto|
+------+--------------+----------+----------+---------------------+
|365   |365           |2025-01-01|2025-12-31|0                    |
+------+--------------+----------+----------+---------------------+



## Chaves nulas na fato

In [62]:
fact_trips.select(
    F.count("*").alias("linhas"),
    F.sum(F.when(F.col("data_id").isNull(), 1).otherwise(0)).alias("data_id_nulo"),
    F.sum(F.when(F.col("clima_id").isNull(), 1).otherwise(0)).alias("clima_id_nulo"),
    F.sum(
        F.when(F.col("localizacao_partida_id").isNull(), 1).otherwise(0)
    ).alias("localizacao_partida_id_nulo"),
    F.sum(
        F.when(F.col("localizacao_chegada_id").isNull(), 1).otherwise(0)
    ).alias("localizacao_chegada_id_nulo"),
).show(truncate=False)

+------+------------+-------------+---------------------------+---------------------------+
|linhas|data_id_nulo|clima_id_nulo|localizacao_partida_id_nulo|localizacao_chegada_id_nulo|
+------+------------+-------------+---------------------------+---------------------------+
|97065 |0           |0            |0                          |0                          |
+------+------------+-------------+---------------------------+---------------------------+



## Integridade dos relacionamentos

In [63]:
orfaos_data = fact_trips.select("data_id").distinct().join(
    dim_data.select("data_id").distinct(), on="data_id", how="left_anti"
).count()

orfaos_clima = fact_trips.select("clima_id").distinct().join(
    dim_clima.select("clima_id").distinct(), on="clima_id", how="left_anti"
).count()

orfaos_partida = fact_trips.select(
    F.col("localizacao_partida_id").alias("localizacao_id")
).distinct().join(
    dim_localizacao.select("localizacao_id").distinct(),
    on="localizacao_id",
    how="left_anti",
).count()

orfaos_chegada = fact_trips.select(
    F.col("localizacao_chegada_id").alias("localizacao_id")
).distinct().join(
    dim_localizacao.select("localizacao_id").distinct(),
    on="localizacao_id",
    how="left_anti",
).count()

spark.createDataFrame(
    [
        ("fact_trips -> dim_data", orfaos_data),
        ("fact_trips -> dim_clima", orfaos_clima),
        ("fact_trips -> dim_localizacao partida", orfaos_partida),
        ("fact_trips -> dim_localizacao chegada", orfaos_chegada),
    ],
    ["relacionamento", "chaves_orfas"],
).show(truncate=False)

+-------------------------------------+------------+
|relacionamento                       |chaves_orfas|
+-------------------------------------+------------+
|fact_trips -> dim_data               |0           |
|fact_trips -> dim_clima              |0           |
|fact_trips -> dim_localizacao partida|0           |
|fact_trips -> dim_localizacao chegada|0           |
+-------------------------------------+------------+



## Amostra enriquecida

In [64]:
fact_trips.alias("f").join(
    dim_data.alias("d"), on="data_id", how="left"
).join(
    dim_clima.alias("c"), on="clima_id", how="left"
).select(
    F.col("f.data_hora_partida"),
    F.col("d.data").alias("data_viagem"),
    F.col("f.duracao_minutos"),
    F.col("f.qtd_passageiros"),
    F.col("f.tipo_pagamento_desc"),
    F.col("f.distancia_km"),
    F.col("f.valor_total"),
    F.col("c.temp_media_c"),
    F.col("c.precipitacao_media_mm"),
    F.col("c.categoria_chuva"),
    F.col("c.registro_clima_incompleto"),
).orderBy("data_hora_partida").show(50, truncate=False)

+-------------------+-----------+---------------+---------------+-------------------+------------+-----------+------------+---------------------+---------------+-------------------------+
|data_hora_partida  |data_viagem|duracao_minutos|qtd_passageiros|tipo_pagamento_desc|distancia_km|valor_total|temp_media_c|precipitacao_media_mm|categoria_chuva|registro_clima_incompleto|
+-------------------+-----------+---------------+---------------+-------------------+------------+-----------+------------+---------------------+---------------+-------------------------+
|2025-01-01 00:00:00|2025-01-01 |63.15          |1              |cartao_credito     |10.3        |74.05      |6.84        |12.4                 |chuva_moderada |false                    |
|2025-01-01 00:00:02|2025-01-01 |9.57           |1              |dinheiro           |2.75        |16.4       |6.84        |12.4                 |chuva_moderada |false                    |
|2025-01-01 00:00:03|2025-01-01 |7.72           |2          

In [65]:
from pyspark.sql import functions as F

print("Contagem das tabelas Gold dev")
for nome, df in {
    "dim_data": dim_data,
    "dim_clima": dim_clima,
    "dim_localizacao": dim_localizacao,
    "fact_trips": fact_trips,
}.items():
    print(nome, df.count())

print("Dim localização enriquecida")
dim_localizacao.orderBy("location_id").show(20, truncate=False)

print("Lookup encontrado ou não encontrado")
dim_localizacao.groupBy("localizacao_sem_lookup").count().show(truncate=False)

print("Chaves nulas na fact")
fact_trips.select(
    F.count("*").alias("total"),
    F.sum(F.col("data_id").isNull().cast("int")).alias("data_id_nulo"),
    F.sum(F.col("clima_id").isNull().cast("int")).alias("clima_id_nulo"),
    F.sum(F.col("localizacao_partida_id").isNull().cast("int")).alias("partida_nula"),
    F.sum(F.col("localizacao_chegada_id").isNull().cast("int")).alias("chegada_nula"),
).show(truncate=False)

print("Chaves órfãs")
orfaos_data = fact_trips.select("data_id").distinct().join(
    dim_data.select("data_id").distinct(), on="data_id", how="left_anti"
).count()

orfaos_clima = fact_trips.select("clima_id").distinct().join(
    dim_clima.select("clima_id").distinct(), on="clima_id", how="left_anti"
).count()

orfaos_partida = fact_trips.select(
    F.col("localizacao_partida_id").alias("localizacao_id")
).distinct().join(
    dim_localizacao.select("localizacao_id").distinct(),
    on="localizacao_id",
    how="left_anti",
).count()

orfaos_chegada = fact_trips.select(
    F.col("localizacao_chegada_id").alias("localizacao_id")
).distinct().join(
    dim_localizacao.select("localizacao_id").distinct(),
    on="localizacao_id",
    how="left_anti",
).count()

print("orfaos_data:", orfaos_data)
print("orfaos_clima:", orfaos_clima)
print("orfaos_partida:", orfaos_partida)
print("orfaos_chegada:", orfaos_chegada)

Contagem das tabelas Gold dev
dim_data 365
dim_clima 365
dim_localizacao 265
fact_trips 97065
Dim localização enriquecida
+--------------+-----------+-------------+-----------------------+------------+----------------------+
|localizacao_id|location_id|borough      |zona                   |zona_servico|localizacao_sem_lookup|
+--------------+-----------+-------------+-----------------------+------------+----------------------+
|1             |1          |EWR          |Newark Airport         |EWR         |false                 |
|2             |2          |Queens       |Jamaica Bay            |Boro Zone   |false                 |
|3             |3          |Bronx        |Allerton/Pelham Gardens|Boro Zone   |false                 |
|4             |4          |Manhattan    |Alphabet City          |Yellow Zone |false                 |
|5             |5          |Staten Island|Arden Heights          |Boro Zone   |false                 |
|6             |6          |Staten Island|Arrochar/For

In [66]:
spark.stop()